# 00 - Setup & smoke test
Runs once per Colab session. **Runtime -> Change runtime type -> T4 GPU** first.
Requires the Colab secret `HF_TOKEN` with *Notebook access* switched on.

## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2. Log in to Hugging Face (token comes from Colab Secrets, never typed here)

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
print("HF user:", whoami()["name"])

## 3. Get the code and install dependencies
If pip upgrades core packages, Colab may ask you to **restart the session**. Do that, then re-run from this cell.

In [ ]:
import os
REPO = "https://github.com/deymonp/sira-lab.git"
if not os.path.exists("/content/sira-lab"):
    !git clone -q {REPO} /content/sira-lab
else:
    !git -C /content/sira-lab pull -q
%cd /content/sira-lab
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## 4. Unit tests (CPU only, a few seconds). Expect `71 passed`.

In [ ]:
!python -m pytest -q

## 5. Smoke test with real (tiny) models
Downloads a ~360M model and prints the most surprising words under it.

In [ ]:
from sira_lab.config import load_config
from sira_lab.scorer import HFScorer
from sira_lab.masker import find_words, score_words

cfg = load_config("configs/smoke.yaml")
scorer = HFScorer(cfg.models.scorer, dtype=cfg.models.dtype)
text = "The proposed method significantly outperforms prior baselines on three benchmark datasets."
ts = scorer.score(text)
words = find_words(text, cfg.mask)
score_words(words, ts.spans, ts.surprisal)
for w in sorted(words, key=lambda w: -w.score):
    print(f"{w.text:>14}  surprisal={w.score:5.2f}  maskable={w.candidate}")

## 6. Record the exact package versions that worked
Copy the printed block back to me (or into `requirements.lock.txt` in the repo) so we can pin it.

In [ ]:
!pip freeze | grep -iE "^(torch|transformers|accelerate|bitsandbytes|sentence-transformers|gradio|numpy|pyyaml|huggingface-hub)==" | tee requirements.lock.txt